# Day 9 — Modules, packages, import system, __main__
Objectives:
- Organize code into modules and packages.
- Use `__name__ == '__main__'`.
- Understand import paths.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-09`. Read
`python/ds-60day/companion-guides/day09_modules_packages.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A module is a Python file loaded into a module object and namespace. An
import locates that code, executes its top level once per interpreter
process, caches the module in `sys.modules`, and binds a name in the
importer. Import-time work should therefore be small and predictable.

A package groups modules under one import namespace. Absolute imports
start at a package available on the import path; relative imports name a
sibling or parent within an already established package context.
`python -m package.module` preserves that context, while directly
executing a nested file often does not. The `if __name__ ==
"__main__":` guard keeps CLI behavior out of ordinary imports.

### Vocabulary

- **module:** a loaded Python file and its namespace.
- **package:** an import namespace containing modules or subpackages.
- **namespace:** a mapping from names to objects.
- **absolute import:** an import written from a top-level package name.
- **relative import:** an import written relative to the current package.
- **entry point:** the deliberate location where execution behavior begins.

## Syntax anatomy

`from statistics import mean` asks the import system for the
`statistics` module and binds only its `mean` attribute locally.
`import statistics as stats` binds the module under `stats`, preserving
visible ownership at each call. In a module, `__name__` equals
`"__main__"` only when that module is the executed entry point.

### Worked example 1 — Compare module and attribute imports

Both styles work; the module-qualified call preserves origin. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import statistics as stats
from statistics import median

values = [2, 3, 100]
(stats.mean(values), median(values), stats.__name__)

**Expected observation:** `(35, 3, 'statistics')` (the mean may display as `35`). The alias still refers to the module object.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Observe the import cache

Repeated imports normally return the same module object. Predict first; then run the next cell.

In [ ]:
import json
import sys

first = json
import json as second
(first is second, sys.modules["json"] is first)

**Expected observation:** `(True, True)`. Python reuses the successfully loaded module from `sys.modules`.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Print `module.__file__` when the wrong same-named module seems to be imported.
2. Run nested package CLIs with `python -m package.module` from the package parent.
3. Inspect `sys.path` as evidence, but do not patch it in lesson code to hide a broken layout.
4. Move shared definitions to a lower-level module when two modules import each other during initialization.

**Alternative to compare:** Import a module when origin clarity matters; re-export a small stable public surface from `__init__.py` when users should not depend on internal layout.

**Boundary to test:** Name shadowing, circular imports, import-time side effects, and direct execution of nested files expose package-context mistakes.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
# Example package layout (refer to project tree rather than running here)
# mypkg/__init__.py
# mypkg/utils.py   -> def add(a,b): return a+b
# main.py          -> from mypkg.utils import add; if __name__=='__main__': print(add(2,3))

import importlib, sys
sys.path[:3]


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Create a `textutils` package containing `slug.py` with `slugify(text: str) -> str`, and expose `slugify` from `textutils/__init__.py`.
   **Expected behavior:** `textutils.slugify(' Data Science ') == 'data-science'`. **Constraints:** keep implementation out of `__init__.py`, use no `sys.path` modification, and test the import from the package's parent directory.
   **Verify:** From the package parent, import `textutils`, assert the slug result, and print `textutils.slugify.__module__` to prove the public name reaches `slug.py`.

2. Add `textutils/cli.py` that uses an absolute import when called from outside and a package-relative import for an internal sibling example. **Run:** `python -m textutils.cli 'Hello World'`. **Expected output:** `hello-world`. **Constraints:** protect CLI execution with `if __name__ == '__main__':` and demonstrate that `import textutils.cli` produces no CLI output.
   **Verify:** Capture `hello-world` from module execution and capture no output from ordinary import in a fresh interpreter.

### Additional mastery practice

A module is executed once per interpreter process and then cached. Package layout and invocation style determine whether imports resolve.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict `__name__` when a file is executed directly versus imported. Which path should run CLI behavior?
   **Progressive hint:** Direct execution uses `__main__`; imports use the module's name.
   **Verify:** Capture `__name__` from direct module execution and import; assert CLI output occurs only in the `__main__` case.
4. **Tracing:** Trace two ordinary imports of the same module and explain the role of `sys.modules` in avoiding repeated top-level execution.
   **Progressive hint:** The module object is cached after its first successful import.
   **Verify:** Use a top-level counter or identity check to prove two imports return the same module object and top-level initialization runs once.
5. **Implementation:** Sketch a `textutils` package exposing `slugify` from `__init__.py` while keeping implementation in `slug.py`.
   **Progressive hint:** The public import surface can be smaller than the package tree.
   **Verify:** From a clean interpreter, assert both `from textutils import slugify` and the internal module call resolve to the same implementation.
6. **Debugging:** Explain why `python textutils/cli.py` can break a relative import and repair the invocation without modifying `sys.path`.
   **Progressive hint:** Run the module from its package parent with `python -m textutils.cli`.
   **Verify:** Show direct nested-file execution fails for the expected package-context reason, then assert `python -m textutils.cli` succeeds without modifying `sys.path`.
7. **Edge case and explanation:** Break a two-module circular import by moving shared types/constants or by passing dependencies explicitly.
   **Progressive hint:** Do not hide the cycle with an unexplained import inside every function.
   **Verify:** Import both modules from a clean process after refactoring; assert neither exposes a partially initialized attribute and describe the removed dependency cycle.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Create a `textutils` package containing `slug.py` with `slugify(text: str) -> str`, and expose `slugify` from `textutils/__init__.py`. **Expected behavior:** `textutils.slugify(' Data Science ') == 'data-science'`. **Constraints:** keep implementation out of `__init__.py`, use no `sys.path` modification, and test the import from the package's parent directory. **Verify:** From the package parent, import `textutils`, assert the slug result, and print `textutils.slugify.__module__` to prove the public name reaches `slug.py`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Create a `textutils` package containing `slug.py` with `slugify(text: str) -> str`, and expose `slugify` from `textutils/__init__.py`. `textutils.slugify(' Data Science ') == 'd...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add `textutils/cli.py` that uses an absolute import when called from outside and a package-relative import for an internal sibling example. **Run:** `python -m textutils.cli 'Hello World'`. **Expected output:** `hello-world`. **Constraints:** protect CLI execution with `if __name__ == '__main__':` and demonstrate that `import textutils.cli` produces no CLI output. **Verify:** Capture `hello-world` from module execution and capture no output from ordinary import in a fresh interpreter.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add `textutils/cli.py` that uses an absolute import when called from outside and a package-relative import for an internal sibling example. `python -m textutils.cli 'Hello World...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict `__name__` when a file is executed directly versus imported. Which path should run CLI behavior? **Progressive hint:** Direct execution uses `__main__`; imports use the module's name. **Verify:** Capture `__name__` from direct module execution and import; assert CLI output occurs only in the `__main__` case.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict `__name__` when a file is executed directly versus imported. Which path should run CLI behavior? Direct execution uses `__main__`; imports use the module's name. Capture...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace two ordinary imports of the same module and explain the role of `sys.modules` in avoiding repeated top-level execution. **Progressive hint:** The module object is cached after its first successful import. **Verify:** Use a top-level counter or identity check to prove two imports return the same module object and top-level initialization runs once.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace two ordinary imports of the same module and explain the role of `sys.modules` in avoiding repeated top-level execution. The module object is cached after its first success...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Sketch a `textutils` package exposing `slugify` from `__init__.py` while keeping implementation in `slug.py`. **Progressive hint:** The public import surface can be smaller than the package tree. **Verify:** From a clean interpreter, assert both `from textutils import slugify` and the internal module call resolve to the same implementation.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Sketch a `textutils` package exposing `slugify` from `__init__.py` while keeping implementation in `slug.py`. The public import surface can be smaller than the package tree. Fro...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Explain why `python textutils/cli.py` can break a relative import and repair the invocation without modifying `sys.path`. **Progressive hint:** Run the module from its package parent with `python -m textutils.cli`. **Verify:** Show direct nested-file execution fails for the expected package-context reason, then assert `python -m textutils.cli` succeeds without modifying `sys.path`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Explain why `python textutils/cli.py` can break a relative import and repair the invocation without modifying `sys.path`. Run the module from its package parent with `python -m...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Break a two-module circular import by moving shared types/constants or by passing dependencies explicitly. **Progressive hint:** Do not hide the cycle with an unexplained import inside every function. **Verify:** Import both modules from a clean process after refactoring; assert neither exposes a partially initialized attribute and describe the removed dependency cycle.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Break a two-module circular import by moving shared types/constants or by passing dependencies explicitly. Do not hide the cycle with an unexplained import inside every function...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
